# Exp1A Formal: Full Disruption Hierarchy — Corrected Marginals

Condition-specific shuffled baselines (1 shuffle) for intact, D6, D7, D4.
wiki_zh + wiki_ja. Reuses existing intact + D4 corrected results.
Only computes D6 + D7 with correction.

Target table:
| Condition | wiki_zh corrected Δ | wiki_ja corrected Δ | % intact |

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import json, math, os, gc, random, time, shutil
from pathlib import Path
from collections import Counter
from scipy import stats
from scipy.ndimage import uniform_filter1d
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from google.colab import drive
drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/LRTIA/Results/Exp1A_formal')
BASE.mkdir(parents=True, exist_ok=True)
DATA = Path('/content/drive/MyDrive/LRTIA/Data')

CORPORA = {
    'wiki_zh': DATA / 'wiki_multilingual/zh_articles.jsonl',
    'wiki_ja': DATA / 'wiki_multilingual/ja_articles.jsonl',
}

MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B'
C = 100
TARGET_LEN = 30
TARGET_FRACS = [0.25, 0.50, 0.75]
MIN_BEFORE = C + 10
N_SHUFFLES = 1
SEED = 20260429

# Check what's already cached
for cn in CORPORA:
    for cond in ['intact', 'D4', 'D6', 'D7']:
        cp = BASE / f'llama_{cn}_{cond}.json'
        if cp.exists():
            with open(cp) as f: n = len(json.load(f))
            print(f'  {cn}/{cond}: cached ({n})')
        else:
            print(f'  {cn}/{cond}: NEEDS COMPUTING')

print(f'\nGPU: {torch.cuda.get_device_name(0)}')
print('Setup done')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16),
    device_map='auto'
)
model.eval()
print('Llama loaded')

In [ ]:
# === Disruption + PPL functions ===

SENT_BOUNDS = set('. ? ! 。 ？ ！ ؟'.split())

def segment_sentences(tids, tok):
    units = []; start = 0
    for i, tid in enumerate(tids):
        if any(ch in SENT_BOUNDS for ch in tok.decode([tid]).strip()):
            units.append((start, i+1)); start = i+1
    if start < len(tids): units.append((start, len(tids)))
    return units

def d6_sentence_swap(ctx, tok):
    units = segment_sentences(ctx, tok)
    if len(units) < 4: return None, len(units), False
    sw = []; i = 0
    while i < len(units)-1: sw.append(units[i+1]); sw.append(units[i]); i += 2
    if i < len(units): sw.append(units[i])
    r = []
    for s,e in sw: r.extend(ctx[s:e])
    return r, len(units), True

def d7_token_swap(ctx, tok):
    units = segment_sentences(ctx, tok)
    r = []
    for s, e in units:
        toks = list(ctx[s:e])
        i = 0
        while i < len(toks)-1: toks[i], toks[i+1] = toks[i+1], toks[i]; i += 2
        r.extend(toks)
    return r, len(units)

@torch.no_grad()
def ppl_nll(ctx_toks, tgt_toks):
    if len(tgt_toks) < 2: return float('inf'), float('inf')
    full = list(ctx_toks) + list(tgt_toks)
    ts = len(ctx_toks)
    ids = torch.tensor([full], device=model.device)
    out = model(ids); logits = out.logits[0]
    nll = 0.0; cnt = 0
    for i in range(ts, len(full)-1):
        lp = torch.log_softmax(logits[i], dim=-1)
        nll += -lp[full[i+1]].item(); cnt += 1
    del out, logits; torch.cuda.empty_cache()
    if cnt == 0: return float('inf'), float('inf')
    mn = nll/cnt; return math.exp(mn), mn

def compute_corrected_curves(cond_ctx, tgt):
    mc = len(cond_ctx)
    o_ppl, o_nll, s_ppl, s_nll = [], [], [], []
    for c in range(mc+1):
        pfx = cond_ctx[-c:] if c > 0 else []
        p, n = ppl_nll(pfx, tgt)
        o_ppl.append(p); o_nll.append(n)
        if c == 0:
            s_ppl.append(p); s_nll.append(n)
        else:
            rng = random.Random(SEED + c)
            sp_l, sn_l = [], []
            for _ in range(N_SHUFFLES):
                sh = list(pfx); rng.shuffle(sh)
                sp, sn = ppl_nll(sh, tgt)
                if not math.isinf(sp): sp_l.append(sp); sn_l.append(sn)
            s_ppl.append(np.mean(sp_l) if sp_l else p)
            s_nll.append(np.mean(sn_l) if sn_l else n)
    dists = list(range(1, mc+1))
    mo = [o_ppl[d-1]-o_ppl[d] for d in dists]
    ms = [s_ppl[d-1]-s_ppl[d] for d in dists]
    delta = [a-b for a,b in zip(mo,ms)]
    return {
        'distances': dists, 'ordered_ppl': o_ppl, 'shuffled_ppl': s_ppl,
        'm_ordered': mo, 'm_shuffled': ms, 'delta_ppl': delta,
    }

# === Quality checks on 5 random targets per corpus ===
print('='*60)
print('QUALITY CHECKS')
print('='*60)

for cn, cp in CORPORA.items():
    docs = []
    with open(cp) as f:
        for line in f: docs.append(json.loads(line))
    
    rng_qc = random.Random(42)
    checked = 0
    for doc in rng_qc.sample(docs, min(5, len(docs))):
        fids = tokenizer.encode(doc['text'], add_special_tokens=False)
        n = len(fids)
        ts = int(n * 0.5)
        te = min(ts + TARGET_LEN, n)
        if ts < MIN_BEFORE: continue
        ctx = fids[ts-C:ts]
        tgt = fids[ts:te]
        
        # D6 checks
        d6_ctx, d6_nu, d6_ok = d6_sentence_swap(ctx, tokenizer)
        if d6_ok:
            assert len(d6_ctx) == C, f'D6 length {len(d6_ctx)}'
            assert Counter(d6_ctx) == Counter(ctx), 'D6 multiset'
            assert not any(t in d6_ctx for t in tgt if ctx.count(t) == 0), 'D6 target leak'
        
        # D7 checks
        d7_ctx, d7_nu = d7_token_swap(ctx, tokenizer)
        assert len(d7_ctx) == C, f'D7 length {len(d7_ctx)}'
        assert Counter(d7_ctx) == Counter(ctx), 'D7 multiset'
        assert not any(t in d7_ctx for t in tgt if ctx.count(t) == 0), 'D7 target leak'
        
        # D7 sentence boundary check: units should be same as intact
        intact_units = segment_sentences(ctx, tokenizer)
        d7_units = segment_sentences(d7_ctx, tokenizer)
        # Note: boundaries may shift slightly due to token swaps moving punctuation
        
        # Baseline check: D6 at c=20 uses same 20 tokens as D6 prefix
        if d6_ok:
            d6_pfx_20 = d6_ctx[-20:]
            assert len(d6_pfx_20) == 20
            # A shuffle of these 20 should contain the same multiset
            sh = list(d6_pfx_20); random.shuffle(sh)
            assert Counter(sh) == Counter(d6_pfx_20), 'D6 shuffle multiset'
        
        d7_pfx_20 = d7_ctx[-20:]
        sh7 = list(d7_pfx_20); random.shuffle(sh7)
        assert Counter(sh7) == Counter(d7_pfx_20), 'D7 shuffle multiset'
        
        checked += 1
    
    print(f'  {cn}: {checked} targets checked — all assertions passed')

print('Quality checks PASSED')
print(f'Functions ready — {N_SHUFFLES} shuffle, condition-specific baselines')

In [ ]:
# === Run D6 + D7 corrected (intact + D4 already cached) ===

d6_stats = {}

for cn, cp in CORPORA.items():
    print(f'\n{"="*60}')
    print(cn)
    print(f'{"="*60}')
    
    docs = []
    with open(cp) as f:
        for line in f: docs.append(json.loads(line))
    
    for cond_name in ['D6', 'D7']:
        cache = BASE / f'llama_{cn}_{cond_name}.json'
        if cache.exists():
            with open(cache) as f: n = len(json.load(f))
            print(f'  {cond_name}: cached ({n})'); continue
        
        t0 = time.time()
        results = []
        n_elig, n_tot = 0, 0
        all_units = []
        
        for doc in tqdm(docs, desc=f'{cn}/{cond_name}'):
            fids = tokenizer.encode(doc['text'], add_special_tokens=False)
            n = len(fids)
            for frac in TARGET_FRACS:
                ts = int(n * frac)
                te = min(ts + TARGET_LEN, n)
                if ts < MIN_BEFORE or te - ts < 5: continue
                ctx = fids[ts-C:ts]
                tgt = fids[ts:te]
                n_tot += 1
                
                if cond_name == 'D6':
                    cond_ctx, nu, ok = d6_sentence_swap(ctx, tokenizer)
                    all_units.append(nu)
                    if not ok: continue
                    n_elig += 1
                elif cond_name == 'D7':
                    cond_ctx, nu = d7_token_swap(ctx, tokenizer)
                    all_units.append(nu)
                    n_elig += 1
                
                assert len(cond_ctx) == C, f'Len {len(cond_ctx)}'
                assert Counter(cond_ctx) == Counter(ctx), 'Multiset'
                
                r = compute_corrected_curves(cond_ctx, tgt)
                r['doc_id'] = doc.get('doc_id', '')
                r['target_frac'] = frac
                r['n_units'] = nu if 'nu' in dir() else 0
                results.append(r)
        
        with open(cache, 'w') as f: json.dump(results, f)
        elapsed = time.time() - t0
        elig = n_elig/n_tot if n_tot > 0 else 0
        print(f'  {cond_name}: {len(results)} results in {elapsed/60:.1f} min')
        print(f'  Eligible: {n_elig}/{n_tot} ({elig:.0%})')
        if cond_name == 'D6':
            d6_stats[cn] = {'elig': elig, 'units': np.mean(all_units), 'n': n_elig}
        if results:
            md = np.mean([np.mean(r['delta_ppl']) for r in results])
            print(f'  Mean corrected Δ: {md:.6f}')

In [ ]:
# === Formal hierarchy table ===
import matplotlib.pyplot as plt

bin_edges = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50, 75, 100]
def fit_pl(marg):
    bm, bc = [], []
    for i in range(len(bin_edges)-1):
        lo, hi = bin_edges[i], bin_edges[i+1]
        vals = marg[lo-1:hi-1]; vals = vals[~np.isnan(vals)]
        if len(vals) > 0 and np.mean(vals) > 0:
            bm.append(np.mean(vals)); bc.append((lo+hi)/2)
    if len(bm) >= 4:
        s, i, r, p, _ = stats.linregress(np.log(bc), np.log(bm))
        return s, r
    return None, None

print(f'\n{"="*80}')
print('FORMAL DISRUPTION HIERARCHY — Corrected Marginals')
print(f'{"="*80}')
print(f'\n{"Corpus":<12} {"Condition":<18} {"Corrected Δ":>12} {"% intact":>10} {"Near":>8} {"Mid":>8} {"Far":>8} {"α":>8} {"r":>8}')
print('-' * 98)

for cn in CORPORA:
    # Get intact value for % calculation
    intact_val = None
    ip = BASE / f'llama_{cn}_intact.json'
    if ip.exists():
        with open(ip) as f: ir = json.load(f)
        intact_curve = np.mean([r['delta_ppl'] for r in ir], axis=0)
        intact_val = np.mean(intact_curve)
    
    for cond in ['intact', 'D6', 'D7', 'D4']:
        cp = BASE / f'llama_{cn}_{cond}.json'
        if not cp.exists():
            print(f'{cn:<12} {cond:<18} {"—":>12}'); continue
        with open(cp) as f: results = json.load(f)
        if not results: continue
        
        curve = np.mean([r['delta_ppl'] for r in results], axis=0)
        total = np.mean(curve)
        near = np.mean(curve[:30])
        mid = np.mean(curve[30:70])
        far = np.mean(curve[70:])
        pct = f'{total/intact_val*100:.0f}%' if intact_val and intact_val > 0 else '—'
        
        alpha, r_val = fit_pl(np.array(curve))
        a_s = f'{alpha:.3f}' if alpha else '—'
        r_s = f'{r_val:.3f}' if r_val else '—'
        
        print(f'{cn:<12} {cond:<18} {total:>12.4f} {pct:>10} {near:>8.4f} {mid:>8.4f} {far:>8.4f} {a_s:>8} {r_s:>8}')
    print()

In [ ]:
# === Visualization: corrected marginal curves + bar plots ===
import matplotlib.pyplot as plt

colors = {'intact': 'blue', 'D6': 'orange', 'D7': 'purple', 'D4': 'green'}

fig, axes = plt.subplots(2, len(CORPORA), figsize=(7*len(CORPORA), 10))
if len(CORPORA) == 1: axes = axes.reshape(-1, 1)

for idx, cn in enumerate(CORPORA):
    # PPL curves
    ax = axes[0, idx]
    for cond in ['intact', 'D6', 'D7', 'D4']:
        cp = BASE / f'llama_{cn}_{cond}.json'
        if not cp.exists(): continue
        with open(cp) as f: r = json.load(f)
        if not r: continue
        ppl = np.mean([x['ordered_ppl'] for x in r], axis=0)
        ax.plot(range(len(ppl)), ppl, color=colors[cond], linewidth=2, label=cond)
    ax.set_title(f'{cn} — Ordered PPL', fontweight='bold')
    ax.set_xlabel('Context length c')
    ax.set_ylabel('Perplexity')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.15)
    
    # Corrected marginals
    ax = axes[1, idx]
    for cond in ['intact', 'D6', 'D7', 'D4']:
        cp = BASE / f'llama_{cn}_{cond}.json'
        if not cp.exists(): continue
        with open(cp) as f: r = json.load(f)
        if not r: continue
        curve = np.mean([x['delta_ppl'] for x in r], axis=0)
        ax.plot(range(1, len(curve)+1), uniform_filter1d(curve, 5),
                color=colors[cond], linewidth=2, label=cond)
    ax.axhline(0, color='gray', linestyle=':', alpha=0.3)
    ax.set_title(f'{cn} — Corrected Marginals', fontweight='bold')
    ax.set_xlabel('Distance d')
    ax.set_ylabel('Δ_d (corrected)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.15)

plt.suptitle('Formal Disruption Hierarchy: Corrected Marginals (condition-specific baselines)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE / 'fig_formal_hierarchy.png', dpi=150, bbox_inches='tight')
plt.show()

# === Bar plot: TotalDelta by condition ===
fig, axes = plt.subplots(1, len(CORPORA), figsize=(6*len(CORPORA), 5))
if len(CORPORA) == 1: axes = [axes]

for idx, cn in enumerate(CORPORA):
    ax = axes[idx]
    cond_names = []
    cond_vals = []
    cond_colors = []
    
    for cond in ['intact', 'D6', 'D7', 'D4']:
        cp = BASE / f'llama_{cn}_{cond}.json'
        if not cp.exists(): continue
        with open(cp) as f: r = json.load(f)
        if not r: continue
        curve = np.mean([x['delta_ppl'] for x in r], axis=0)
        cond_names.append(cond)
        cond_vals.append(np.mean(curve))
        cond_colors.append(colors[cond])
    
    bars = ax.bar(range(len(cond_names)), cond_vals, color=cond_colors, alpha=0.7, edgecolor='black')
    ax.set_xticks(range(len(cond_names)))
    ax.set_xticklabels(cond_names, fontsize=11)
    ax.set_ylabel('Mean Corrected Marginal Δ', fontsize=11)
    ax.set_title(cn, fontweight='bold', fontsize=13)
    ax.axhline(0, color='gray', linestyle=':', alpha=0.3)
    ax.grid(True, alpha=0.15, axis='y')
    
    for i, v in enumerate(cond_vals):
        pct = f'{v/cond_vals[0]*100:.0f}%' if cond_vals[0] > 0 else ''
        ax.text(i, v + 0.005, f'{v:.3f}\n({pct})', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Disruption Hierarchy: Mean Corrected Marginal by Condition',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE / 'fig_formal_hierarchy_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figures saved')